### Earthquake PINN Training on Colab GPU
This notebook clones the repository, installs dependencies, and runs the full-scale training.
It also includes optional Hyperparameter Optimization (Optuna).

**Note**: Make sure you have pushed your local changes (`git push origin dev`) before running this.

In [22]:
!nvidia-smi

Sun Feb 15 23:57:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [23]:
# Install uv and clean up existing clone
!pip install uv
!rm -rf earthquake_proj
!git clone -b dev --single-branch https://github.com/sattary/earthquake_proj.git
%cd earthquake_proj

Cloning into 'earthquake_proj'...
remote: Enumerating objects: 157, done.
remote: Counting objects: 100% (157/157), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 157 (delta 54), reused 149 (delta 46), pack-reused 0 (from 0)
Receiving objects: 100% (157/157), 20.22 MiB | 14.70 MiB/s, done.
Resolving deltas: 100% (54/54), done.
/content/earthquake_proj/earthquake_proj/earthquake_proj/earthquake_proj


In [24]:
# Sync environment
!uv sync

Using CPython 3.12.12 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 100 packages in 0.89ms
Installed 94 packages in 427ms                              
 + alembic==1.18.4
 + annotated-doc==0.0.4
 + asttokens==3.0.1
 + certifi==2026.1.4
 + click==8.3.1
 + cloudpickle==3.1.2
 + colorlog==6.10.1
 + comm==0.2.3
 + contourpy==1.3.3
 + cuda-bindings==12.9.4
 + cuda-pathfinder==1.3.4
 + cycler==0.12.1
 + debugpy==1.8.20
 + decorator==5.2.1
 + executing==2.2.1
 + filelock==3.24.0
 + fonttools==4.61.1
 + fsspec==2026.2.0
 + greenlet==3.3.1
 + ipykernel==7.2.0
 + ipython==9.10.0
 + ipython-pygments-lexers==1.1.1
 + jedi==0.19.2
 + jinja2==3.1.6
 + joblib==1.5.3
 + jupyter-client==8.8.0
 + jupyter-core==5.9.1
 + kiwisolver==1.4.9
 + llvmlite==0.46.0
 + mako==1.3.10
 + markdown-it-py==4.0.0
 + markupsafe==3.0.3
 + matplotlib==3.10.8
 + matplotlib-inline==0.2.1
 + mdurl==0.1.2
 + mpmath==1.3.0
 + nest-asyncio==1.6.0
 + networkx==3.6.1
 + numba==0.63.1
 + numpy==2.

In [25]:
import os
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("results/tables", exist_ok=True)
os.makedirs("results/figs", exist_ok=True)

# Ensure package initialization for -m calls
if os.path.exists('src'):
    if not os.path.exists('src/__init__.py'):
        open('src/__init__.py', 'a').close()
        print("Initialized 'src' as package.")
elif os.path.exists('earthquake_proj'):
    if not os.path.exists('earthquake_proj/__init__.py'):
        open('earthquake_proj/__init__.py', 'a').close()
        print("Initialized 'earthquake_proj' as package.")

Initialized 'src' as package.


In [33]:
!uv run python -m src.pinn.tune --help'

/bin/bash: -c: line 1: unexpected EOF while looking for matching `''
/bin/bash: -c: line 2: syntax error: unexpected end of file


In [30]:
# Step 1: Tune Hyperparameters (Optional - skip if using defaults)
!if [ -d "src" ]; then PKG="src"; else PKG="earthquake_proj"; fi; \
export PYTHONPATH=$PYTHONPATH:$(pwd); \
echo "Starting Hyperparameter Tuning with Optuna..."; \
uv run python -m $PKG.pinn.tune --trials 50 --epochs 500 --spatial-dim 3 --velocity-file data/Morteza_2023/Vel/Pwave.3D.txt

Starting Hyperparameter Tuning with Optuna...
Usage: python -m src.pinn.tune [OPTIONS]
Try 'python -m src.pinn.tune --help' for help.
╭─ Error ──────────────────────────────────────────────────────────────────────╮
│ No such option: --trials Did you mean --n-trials?                            │
╰──────────────────────────────────────────────────────────────────────────────╯


In [28]:
# Step 2: Train with Best Params Found in Step 1
!if [ -d "src" ]; then PKG="src"; else PKG="earthquake_proj"; fi; \
export PYTHONPATH=$PYTHONPATH:$(pwd); \
uv run python -m $PKG.pinn.cli train --config results/tables/best_params.json --epochs 10000 --n-coll 10000 --spatial-dim 3 --velocity-file data/Morteza_2023/Vel/Pwave.3D.txt --gpu

╭───────────────────── Traceback (most recent call last) ──────────────────────╮
│ /content/earthquake_proj/earthquake_proj/earthquake_proj/earthquake_proj/src │
│ /pinn/cli.py:38 in train                                                     │
│                                                                              │
│    35 │   import json                                                        │
│    36 │                                                                      │
│    37 │   if config:                                                         │
│ ❱  38 │   │   with open(config, "r") as f:                                   │
│    39 │   │   │   params = json.load(f)                                      │
│    40 │   │   │   typer.echo(f"Loading config from {config}: {params}")      │
│    41 │   │   │   # Override defaults with config                            │
╰──────────────────────────────────────────────────────────────────────────────╯
FileNotFoundError: [Errno 2]

In [ ]:
# Generate Visualizations
!if [ -d "src" ]; then PKG="src"; else PKG="earthquake_proj"; fi; \
export PYTHONPATH=$PYTHONPATH:$(pwd); \
uv run python -m $PKG.pinn.visualize --model-path checkpoints/final_model.pth --depth 10.0 --output-stress results/figs/stress_map_10km.png --output-velocity results/figs/velocity_map_10km.png

In [ ]:
from IPython.display import Image, display
import os
for f in ['results/figs/stress_map_10km.png', 'results/figs/velocity_map_10km.png']:
    if os.path.exists(f): display(Image(f))

In [ ]:
# Pack Everything for Download/Active Storage
!zip -r results_archive.zip results/ checkpoints/final_model.pth
try:
  from google.colab import files
  files.download('results_archive.zip')
except ImportError:
  print('Not running on Google Colab, skipping download.')